# 01 — Segmentation visual audit

Runs Silero segmentation and audits the exact original per-recording frame, segment, and four-panel figure artifacts.

Every displayed denominator and paper-facing visual is also saved under separate `figures/` and `tables/` folders within `outputs/visualization/`. Empty or under-supported analyses remain visible as audit rows; they are never silently removed. The final cell explains whether the next stage is allowed.

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, Markdown, display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paper1_qc").exists():
            return candidate
    raise FileNotFoundError("Open Jupyter from inside the paper_1 project.")

ROOT = find_project_root()
CONFIG = ROOT / "config" / "project.yaml"
OUTPUT = ROOT / "outputs"
MAIN_OUTPUTS = ROOT / "MAIN outputs"
VIZ_ROOT = OUTPUT / "visualization"
sys.path.insert(0, str(ROOT / "src"))

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def read_stage(relative_without_suffix):
    stem = OUTPUT / relative_without_suffix
    parquet = stem.with_suffix(".parquet")
    csv = stem.with_suffix(".csv")
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        try:
            return pd.read_csv(csv)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
    raise FileNotFoundError(f"Missing required stage table: {parquet} or {csv}")

def read_optional_stage(relative_without_suffix):
    stem = OUTPUT / relative_without_suffix
    if stem.with_suffix(".parquet").exists() or stem.with_suffix(".csv").exists():
        return read_stage(relative_without_suffix)
    print("OPTIONAL TABLE NOT AVAILABLE:", relative_without_suffix)
    return pd.DataFrame()

def run_cli(*arguments):
    command = [sys.executable, "-m", "paper1_qc.cli", "--config", str(CONFIG), *arguments]
    print("RUN:", " ".join(map(str, command)))
    subprocess.run(command, cwd=ROOT, check=True)

def save_table(frame, folder, name):
    target = VIZ_ROOT / folder / "tables"
    target.mkdir(parents=True, exist_ok=True)
    path = target / f"{name}.csv"
    frame.to_csv(path, index=False)
    print("TABLE:", path.relative_to(ROOT), f"({len(frame):,} rows)")
    return path

def save_figure(fig, folder, name):
    target = VIZ_ROOT / folder / "figures"
    target.mkdir(parents=True, exist_ok=True)
    png = target / f"{name}.png"
    svg = target / f"{name}.svg"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("FIGURE:", png.relative_to(ROOT))
    return png, svg

def stage_gate(stage_name, can_continue, reasons, next_step):
    status = "PASS — safe to continue" if can_continue else "BLOCKED — decision/action required"
    color = "#1B7F3A" if can_continue else "#B22222"
    details = "\n".join(f"- {reason}" for reason in reasons) if reasons else "- No blocking findings."
    display(Markdown(
        f"### {stage_name}: <span style='color:{color}'>{status}</span>\n\n"
        f"{details}\n\n**Next step:** {next_step}"
    ))
    return can_continue

assert CONFIG.exists(), "Copy config/project.example.yaml to config/project.yaml and review it."
print("Project:", ROOT)
print("Config:", CONFIG)
print("MAIN outputs:", MAIN_OUTPUTS)
print("Visualization outputs:", VIZ_ROOT)


The visible Silero artifacts in this notebook reproduce the original pipeline: one 30-ms frame CSV, one segment CSV, and one four-panel PNG per recording. All of them now live inside `outputs/01_segmentation`. The aggregate raw/primary/strict-speech/guarded-nonspeech interval table retains unpadded sample-index boundaries; the 30-ms artifact layer is visualization only. A separate boundary-audit CSV/PNG quantifies display binning and local edge evidence. Unusual ALS speech is not invalid merely because it is fragmented by VAD.

In [ ]:
RUN_SEGMENTATION = False
if RUN_SEGMENTATION:
    run_cli("segment")
else:
    print("Using existing segmentation outputs.")


In [ ]:
segments = read_stage("01_segmentation/bamboo_segmentation_intervals")
errors = read_stage("01_segmentation/segmentation_errors")

required = {"file_name", "profile", "view", "start_sec", "end_sec", "duration_sec"}
missing = required - set(segments.columns)
assert not missing, f"Segmentation table is missing columns: {sorted(missing)}"
assert (segments["end_sec"] >= segments["start_sec"]).all(), "Negative interval detected"
assert np.allclose(
    segments["duration_sec"],
    segments["end_sec"] - segments["start_sec"],
    rtol=0,
    atol=1e-6,
), "Saved duration does not equal end-start"

summary = (
    segments.groupby(["profile", "view"], as_index=False)
    .agg(
        recordings=("file_name", "nunique"),
        intervals=("duration_sec", "size"),
        total_duration_sec=("duration_sec", "sum"),
        median_interval_sec=("duration_sec", "median"),
        q05_interval_sec=("duration_sec", lambda x: x.quantile(.05)),
        q95_interval_sec=("duration_sec", lambda x: x.quantile(.95)),
    )
)
save_table(summary, "01_segmentation", "interval_summary_by_profile_and_view")
display(summary)
display(errors.head(50))


In [ ]:
# One row per recording/profile: denominators, interval counts, and speech fractions.
recording_duration = (
    segments.groupby(["file_name", "profile"], as_index=False)["end_sec"]
    .max().rename(columns={"end_sec": "recording_duration_sec"})
)
duration_wide = (
    segments.pivot_table(
        index=["file_name", "profile"],
        columns="view",
        values="duration_sec",
        aggfunc="sum",
        fill_value=0,
    ).reset_index()
)
count_wide = (
    segments.pivot_table(
        index=["file_name", "profile"],
        columns="view",
        values="duration_sec",
        aggfunc="size",
        fill_value=0,
    ).add_prefix("n_intervals__").reset_index()
)
recording = recording_duration.merge(duration_wide, on=["file_name", "profile"]).merge(
    count_wide, on=["file_name", "profile"]
)
for view in ["raw_speech", "primary_speech", "strict_speech", "strict_internal_nonspeech"]:
    if view not in recording:
        recording[view] = 0.0
    recording[f"fraction__{view}"] = recording[view] / recording["recording_duration_sec"]
save_table(recording, "01_segmentation", "recording_profile_support")
display(recording.describe(include="all").T)


In [ ]:
LEGACY = OUTPUT / "01_segmentation" / "segmentation" / "silero"
LEGACY_FIGURES = OUTPUT / "01_segmentation" / "figures" / "segmentation" / "silero"
legacy_summary = pd.read_csv(LEGACY / "summary" / "silero_all_summary.csv")

artifact_audit = pd.DataFrame([{
    "summary_rows": len(legacy_summary),
    "segment_csv_files": len(list((LEGACY / "segments").glob("*_segments.csv"))),
    "frame_csv_files": len(list((LEGACY / "frames").glob("*_frames.csv"))),
    "accepted_png_files": len(list((LEGACY_FIGURES / "accepted").glob("*_silero.png"))),
    "flagged_png_files": len(list((LEGACY_FIGURES / "flagged").glob("*_silero.png"))),
    "excluded_png_files": len(list((LEGACY_FIGURES / "excluded").glob("*_silero.png"))),
    "boundary_audit_csv_files": len(list((LEGACY / "boundary_audit").glob("*_boundary_audit.csv"))),
    "boundary_audit_png_files": len(list((LEGACY_FIGURES / "boundary_audit").glob("*_boundary_audit.png"))),
}])
artifact_audit["total_png_files"] = artifact_audit[
    ["accepted_png_files", "flagged_png_files", "excluded_png_files"]
].sum(axis=1)
artifact_audit["one_segment_file_per_summary_row"] = (
    artifact_audit["segment_csv_files"] == artifact_audit["summary_rows"]
)
artifact_audit["one_frame_file_per_summary_row"] = (
    artifact_audit["frame_csv_files"] == artifact_audit["summary_rows"]
)
artifact_audit["one_figure_per_summary_row"] = (
    artifact_audit["total_png_files"] == artifact_audit["summary_rows"]
)
artifact_audit["one_boundary_audit_csv_per_summary_row"] = (
    artifact_audit["boundary_audit_csv_files"] == artifact_audit["summary_rows"]
)
artifact_audit["one_boundary_audit_png_per_summary_row"] = (
    artifact_audit["boundary_audit_png_files"] == artifact_audit["summary_rows"]
)
save_table(artifact_audit, "01_segmentation", "legacy_artifact_completeness")
display(artifact_audit)
assert artifact_audit[[
    "one_segment_file_per_summary_row",
    "one_frame_file_per_summary_row",
    "one_figure_per_summary_row",
    "one_boundary_audit_csv_per_summary_row",
    "one_boundary_audit_png_per_summary_row",
]].all(axis=None), "Silero per-recording artifact contract is incomplete"


In [ ]:
# Inspect the exact original-format per-recording CSVs and PNG.
EXAMPLE_FILE = None  # replace with an exact filename, or leave None for the first row
candidate = EXAMPLE_FILE or legacy_summary["file_name"].dropna().iloc[0]
candidate_row = legacy_summary.loc[legacy_summary["file_name"].eq(candidate)].iloc[0]
candidate_stem = Path(candidate).stem
candidate_frames = pd.read_csv(LEGACY / "frames" / f"{candidate_stem}_frames.csv")
candidate_segments = pd.read_csv(LEGACY / "segments" / f"{candidate_stem}_segments.csv")

expected_frame_columns = [
    "frame_idx", "mid_sec", "rms", "rms_db", "speech_vad_raw",
    "speech_vad_smooth", "speech_mask_strict", "nonspeech_mask_strict",
    "threshold", "frame_ms",
]
expected_segment_columns = [
    "segment_type", "start_sec", "end_sec", "duration_sec",
    "run_start_frame", "run_end_frame", "segment_role",
]
assert candidate_frames.columns.tolist() == expected_frame_columns
assert candidate_segments.columns.tolist() == expected_segment_columns
assert set(candidate_segments["segment_role"]).issubset({
    "speech", "leading_nonspeech", "internal_nonspeech", "trailing_nonspeech"
})

display(Markdown(f"### Original-format artifact audit: `{candidate}`"))
display(candidate_segments)
display(candidate_frames.head(20))
display(Image(filename=str(candidate_row["plot_path"]), width=1100))


In [ ]:
# Exact-boundary audit: review evidence only; no energy-based auto-snapping.
resolved_parameters = json.loads(
    (OUTPUT / "01_segmentation" / "logs" / "silero_segmentation_config.json")
    .read_text(encoding="utf-8")
)
display(pd.DataFrame([resolved_parameters]).T.rename(columns={0: "resolved_value"}))

boundary_summary = read_stage("01_segmentation/bamboo_segmentation_summary")[[
    "file_name", "qc_status", "boundary_edges", "boundary_low_contrast_edges",
    "boundary_low_contrast_fraction", "boundary_min_contrast_db",
    "boundary_audit_path", "boundary_plot_path",
]]
save_table(boundary_summary, "01_segmentation", "boundary_alignment_summary")
display(boundary_summary.sort_values(
    ["boundary_low_contrast_fraction", "boundary_min_contrast_db"],
    ascending=[False, True],
).head(40))

BOUNDARY_EXAMPLE_FILE = None
if BOUNDARY_EXAMPLE_FILE:
    boundary_row = boundary_summary.loc[
        boundary_summary["file_name"].eq(BOUNDARY_EXAMPLE_FILE)
    ].iloc[0]
else:
    boundary_row = boundary_summary.sort_values(
        ["boundary_low_contrast_fraction", "boundary_min_contrast_db"],
        ascending=[False, True],
    ).iloc[0]
display(pd.read_csv(boundary_row["boundary_audit_path"]))
display(Image(filename=str(boundary_row["boundary_plot_path"]), width=1100))


In [ ]:
# Preserve the original pipeline's accepted/flagged/excluded visual audit.
qc_summary = read_stage("01_segmentation/bamboo_segmentation_summary")
qc_counts = (
    qc_summary["qc_status"].value_counts(dropna=False)
    .rename_axis("qc_status").reset_index(name="logical_recordings")
)
qc_counts["percent"] = 100 * qc_counts["logical_recordings"] / max(1, len(qc_summary))
save_table(qc_summary, "01_segmentation", "recording_level_silero_qc")
save_table(qc_counts, "01_segmentation", "accepted_flagged_excluded_counts")
display(qc_counts)

for status in ["accepted", "flagged", "excluded"]:
    subset = qc_summary.loc[qc_summary["qc_status"].eq(status)]
    if subset.empty:
        print(f"No {status} recording exists.")
        continue
    example_path = Path(str(subset.iloc[0]["plot_path"]))
    display(Markdown(f"**{status.upper()} example:** `{subset.iloc[0]['file_name']}`"))
    if example_path.exists():
        display(Image(filename=str(example_path), width=1000))
    else:
        print("Diagnostic path is missing:", example_path)


## Required segmentation decision

Non-outlying accepted recordings default to `KEEP + AUTO`. Every flagged/excluded recording and every accepted segmentation-only outlier requires review. The widget shows all recordings in a scrollable/searchable browser and supports audio playback, the original four-panel plot, one-click `KEEP + AUTO`, `KEEP + MANUAL`, and `EXCLUDE + NONE`. Do not exclude unusual ALS speech merely because VAD fragmented it, and do not edit boundaries to remove acoustic noise. `Task Completed as Instructed = NO` is a locked automatic exclusion.

In [ ]:
run_cli("segment-template")
import yaml

project_cfg = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))
adjudication_path = ROOT / project_cfg.get("data_freeze", {}).get(
    "segmentation_adjudication", "config/segmentation_adjudication.csv"
)
manual_override_path = ROOT / project_cfg.get("data_freeze", {}).get(
    "manual_segmentation_overrides", "config/manual_segmentation_overrides.csv"
)
review = pd.read_csv(adjudication_path, keep_default_na=False)
from paper1_qc.segmentation import segmentation_pending_reviews

pending_review = segmentation_pending_reviews(review)
save_table(pending_review, "01_segmentation", "pending_segmentation_decisions")
display(pending_review[[
    "file_name", "automatic_qc_status", "task_completed_as_instructed",
    "automatic_task_exclusion", "accepted_outlier", "review_reasons",
    "decision", "boundary_source", "reviewer", "review_date", "notes"
]])

selection_summary = (
    review.groupby(
        [
            "automatic_qc_status", "automatic_task_exclusion",
            "accepted_outlier", "review_required",
        ],
        dropna=False,
    ).size().rename("logical_recordings").reset_index()
)
save_table(selection_summary, "01_segmentation", "review_selection_summary")
display(selection_summary)


In [ ]:
from paper1_qc.config import load_config, resolve_executable
from paper1_qc.segmentation_review import launch_segmentation_review_widget

cfg = load_config(CONFIG)
DEFAULT_REVIEWER = ""  # enter your name once
review_widget = launch_segmentation_review_widget(
    summary=qc_summary,
    automatic_intervals=segments,
    review_path=adjudication_path,
    overrides_path=manual_override_path,
    default_reviewer=DEFAULT_REVIEWER,
    ffmpeg=resolve_executable(cfg["software"]["ffmpeg"], "ffmpeg"),
    ffprobe=resolve_executable(cfg["software"]["ffprobe"], "ffprobe"),
)
display(review_widget)


In [ ]:
# Run after the interactive review; the widget writes decisions to disk.
review = pd.read_csv(adjudication_path, keep_default_na=False)
pending_review = segmentation_pending_reviews(review)
display(pending_review[[
    "file_name", "automatic_qc_status", "task_completed_as_instructed",
    "review_reasons", "decision", "boundary_source", "reviewer",
    "review_date", "notes"
]])

RUN_SEGMENTATION_ADJUDICATION = False
if RUN_SEGMENTATION_ADJUDICATION:
    assert pending_review.empty, "Complete every required review before freezing."
    run_cli("segment-adjudicate")
else:
    print("Set RUN_SEGMENTATION_ADJUDICATION=True only after pending_review is empty.")


In [ ]:
segmentation_freeze_version = project_cfg.get("segmentation_freeze", {}).get(
    "version",
    project_cfg.get("data_freeze", {}).get("version", "v1"),
)
SEGMENTATION_FREEZE = (
    MAIN_OUTPUTS / "01_SEGMENTATION_FREEZE" / str(segmentation_freeze_version)
)
frozen_decision_path = SEGMENTATION_FREEZE / "frozen_segmentation_decisions.csv"
frozen_interval_path = SEGMENTATION_FREEZE / "frozen_segmentation_intervals.csv"
reviewed_summary_path = (
    OUTPUT / "01_segmentation_after_review" / "segmentation" / "silero"
    / "summary" / "silero_after_review_summary.csv"
)
decision_exists = frozen_decision_path.exists()
interval_exists = frozen_interval_path.exists()
reviewed_exists = reviewed_summary_path.exists()
segmentation_reasons = []
if not pending_review.empty:
    segmentation_reasons.append(
        f"{len(pending_review)} required/incomplete segmentation reviews remain."
    )
if not decision_exists:
    segmentation_reasons.append("Frozen segmentation decision table does not exist.")
if not interval_exists:
    segmentation_reasons.append("Frozen segmentation interval table does not exist.")
if not reviewed_exists:
    segmentation_reasons.append("Post-review segmentation summary does not exist.")
segmentation_ready = stage_gate(
    "Silero segmentation",
    not segmentation_reasons,
    segmentation_reasons,
    "Open 02_goal1_occurrence_and_acquisition_variability.ipynb only after this gate passes.",
)